# LeetCode 494 — Target Sum bằng Python

Link đề: https://leetcode.com/problems/target-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Backtracking, 0-1 Knapsack

Skeleton LeetCode:

```python
class Solution:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc; mọi đáp án trong notebook được chạy lại và đối chiếu với một oracle vét cạn độc lập (thử mọi cách gán dấu).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đặt dấu `+` hoặc `-` trước **mỗi** số của `nums` sao cho biểu thức có giá trị đúng bằng `target`, và **đếm số cách** gán dấu.
2. Nắm công cụ Python: `itertools.product` để sinh mọi tổ hợp dấu, đệ quy, `functools.lru_cache` (memoization), `dict.get`, `collections.defaultdict` / `Counter`, toán tử `%` và `//` để kiểm tra tính chẵn lẻ.
3. Đi từ **vét cạn `2^n`** → **đệ quy `dfs(i, tổng)`** → **đệ quy có nhớ (memo)** → **đổi bài toán sang "đếm tập con có tổng P" (knapsack 0-1 một chiều)** → **DP bằng từ điển "tổng → số cách"**.
4. Hiểu mấu chốt đại số: gọi `P` là tổng các số được gán `+`, `N` là tổng các số được gán `-`; khi đó `P - N = target` và `P + N = S` (tổng tất cả), suy ra `P = (S + target) / 2` — bài toán biến thành "có bao nhiêu tập con có tổng `P`".
5. Tránh các lỗi kinh điển: duyệt `j` **xuôi** ở knapsack 1 chiều (dùng một số nhiều lần), quên kiểm tra chẵn lẻ / `|target| > S`, bỏ qua số `0` (mỗi số `0` nhân đôi số cách), memo thiếu khóa `tổng`, nhầm "đếm số cách" với "có tồn tại cách hay không".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `target`.

Ta xây một **biểu thức** từ `nums` bằng cách đặt **một** ký hiệu `+` hoặc `-` trước **mỗi** số rồi nối chúng lại. Ví dụ `nums = [2, 1]`: đặt `+` trước `2` và `-` trước `1` ta được biểu thức `"+2-1"`.

Trả về **số biểu thức khác nhau** có giá trị bằng `target`.

**Ví dụ 1:** `nums = [1,1,1,1,1]`, `target = 3` → `5`. Có 5 cách: `-1+1+1+1+1`, `+1-1+1+1+1`, `+1+1-1+1+1`, `+1+1+1-1+1`, `+1+1+1+1-1`.

**Ví dụ 2:** `nums = [1]`, `target = 1` → `1`.

**Ràng buộc:**

- `1 <= nums.length <= 20`
- `0 <= nums[i] <= 1000`
- `0 <= sum(nums[i]) <= 1000`
- `-1000 <= target <= 1000`

In [1]:
import random
import time
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import product
from typing import List

examples = [
    (([1, 1, 1, 1, 1], 3), 5),
    (([1], 1), 1),
]
for (nums, target), expected in examples:
    print("nums =", nums, "target =", target, "->", expected)

nums = [1, 1, 1, 1, 1] target = 3 -> 5
nums = [1] target = 1 -> 1


### 2.1. Đọc kỹ đề

- **Mỗi số đều phải có dấu** (không được bỏ số nào). Số cách tổng cộng là `2^n`; ta chỉ đếm những cách cho đúng `target`.
- **Hai biểu thức khác nhau** nếu dấu ở **vị trí** nào đó khác nhau, dù giá trị các số bằng nhau. Trong ví dụ 1 có 5 số `1` giống hệt nhau mà vẫn có 5 cách vì vị trí số mang dấu `-` khác nhau.
- `nums[i] = 0` được phép: `+0` và `-0` là **hai** biểu thức khác nhau nhưng cùng giá trị ⇒ mỗi số `0` làm số cách **nhân đôi**.
- `sum(nums) <= 1000` ⇒ mọi giá trị trung gian nằm trong `[-1000, 1000]`; đây là gợi ý để DP theo "giá trị tổng" có kích thước nhỏ.
- `n <= 20` ⇒ `2^20 ≈ 10^6`: vét cạn vẫn có thể chạy được (hơi chậm với Python), nhưng đề cố tình để mở đường cho cả DP.

## 3. Kiến thức nền

### 3.1. `itertools.product` — sinh mọi tổ hợp dấu

`product([1, -1], repeat=n)` sinh **mọi bộ `n` dấu** (mỗi vị trí là `+1` hoặc `-1`), tổng cộng `2^n` bộ. Ta nhân từng dấu với số tương ứng để tính giá trị biểu thức.

In [2]:
nums = [2, 1]
for signs in product([1, -1], repeat=len(nums)):
    value = sum(s * x for s, x in zip(signs, nums))
    expr = "".join(("+" if s == 1 else "-") + str(x) for s, x in zip(signs, nums))
    print(signs, expr, "=", value)

(1, 1) +2+1 = 3
(1, -1) +2-1 = 1
(-1, 1) -2+1 = -1
(-1, -1) -2-1 = -3


### 3.2. Đệ quy: bài toán nhỏ hơn cùng dạng

`dfs(i, cur)` = "đã quyết định dấu cho `i` số đầu, tổng hiện tại là `cur`; còn bao nhiêu cách hoàn tất để được `target`?". Với số thứ `i` ta có hai lựa chọn `cur + nums[i]` hoặc `cur - nums[i]`. Điều kiện dừng khi `i == n`: được 1 cách nếu `cur == target`, ngược lại 0.

### 3.2b. Memoization (`lru_cache`)

Nếu cùng một cặp `(i, cur)` được gọi nhiều lần, kết quả luôn như nhau, nên ta **nhớ** lại để khỏi tính lại. `functools.lru_cache` làm việc này tự động cho hàm có tham số băm được.

In [3]:
@lru_cache(maxsize=None)
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

calls = 0
print(fib(30), "số lần thực sự tính:", calls)     # chỉ 31 lần thay vì ~2.7 triệu
fib.cache_clear()

832040 số lần thực sự tính: 31


### 3.3. Từ điển đếm: `dict.get`, `defaultdict`, `Counter`

Để đếm "tổng `s` có bao nhiêu cách" ta dùng từ điển `ways = {s: số cách}`. `ways.get(s, 0)` trả `0` nếu `s` chưa có; `defaultdict(int)` tự khởi tạo `0`.

In [4]:
ways = {}
ways[3] = ways.get(3, 0) + 1
ways[3] = ways.get(3, 0) + 1
print(ways)                     # {3: 2}

dd = defaultdict(int)
dd[-1] += 4
dd[5] += 1
print(dict(dd))
print(Counter([1, 1, 2]))

{3: 2}
{-1: 4, 5: 1}
Counter({1: 2, 2: 1})


### 3.4. `%` và `//` để kiểm tra chẵn lẻ và chia đôi

`x % 2 == 0` nghĩa là `x` chẵn. Nếu `x` chẵn thì `x // 2` chia đúng (không mất phần dư). Điều này sẽ cần khi ta tính `(S + target) / 2` ở Cách 4. Lưu ý với số **âm**, Python vẫn cho `-3 % 2 == 1` (kết quả luôn không âm khi chia cho số dương).

In [5]:
for x in [8, 7, -3, -4]:
    print(f"x={x:>2}: x % 2 = {x % 2}, x // 2 = {x // 2}")

x= 8: x % 2 = 0, x // 2 = 4
x= 7: x % 2 = 1, x // 2 = 3
x=-3: x % 2 = 1, x // 2 = -2
x=-4: x % 2 = 0, x // 2 = -2


### 3.5. Bảng `dp` một chiều và vòng lặp `range` ngược

`range(a, b, -1)` đi từ `a` xuống `b + 1`. Ở bài knapsack 0-1 ta cần duyệt mảng `dp` **từ phải sang trái** để mỗi số chỉ được dùng **một lần**. Ta sẽ thấy lý do ở mục 7 (Cách 4) và lỗi ở mục 13.

In [6]:
print(list(range(4, 0, -1)))        # [4, 3, 2, 1]
print(list(range(4, 1 - 1, -1)))    # [4, 3, 2, 1]  (tới x = 1 thì dừng sau khi lấy 1)

[4, 3, 2, 1]
[4, 3, 2, 1]


## 4. Ý tưởng nền tảng: tính tay

Lấy `nums = [1, 2, 3]`, `target = 0`. Tổng `S = 6`. Có `2^3 = 8` cách gán dấu:

| dấu | biểu thức | giá trị |
|---|---|---|
| `+ + +` | `+1+2+3` | 6 |
| `+ + -` | `+1+2-3` | **0** |
| `+ - +` | `+1-2+3` | 2 |
| `+ - -` | `+1-2-3` | -4 |
| `- + +` | `-1+2+3` | 4 |
| `- + -` | `-1+2-3` | -2 |
| `- - +` | `-1-2+3` | **0** |
| `- - -` | `-1-2-3` | -6 |

Có **2** cách cho `target = 0`. Nhận xét: hai cách đó là **đối xứng nhau**: `+1+2-3` và `-1-2+3` (đổi mọi dấu). Mỗi biểu thức chia các số thành hai nhóm: nhóm mang `+` (`{1,2}`) có tổng `3`, nhóm mang `-` (`{3}`) có tổng `3`. Vì `P - N = 0` và `P + N = 6` ⇒ `P = 3`: ta đang đếm **tập con có tổng 3**: `{1,2}` và `{3}` ⇒ 2 tập con ⇒ 2 cách. Mỗi tập con "mang `+`" tương ứng đúng một cách gán dấu.

> Trực giác then chốt: **chọn dấu cho từng số = chọn một tập con các số mang `+`**.

In [7]:
nums, target = [1, 2, 3], 0
S = sum(nums)
P = (S + target) // 2
print("S =", S, " P =", P)
subsets = []
for mask in range(1 << len(nums)):
    chosen = [nums[i] for i in range(len(nums)) if (mask >> i) & 1]
    if sum(chosen) == P:
        subsets.append(chosen)
print("tập con có tổng P:", subsets, "->", len(subsets), "cách")

S = 6  P = 3
tập con có tổng P: [[1, 2], [3]] -> 2 cách


## 5. Cách 1 — Brute Force: thử mọi cách gán dấu

**Ý tưởng.** Với mỗi bộ dấu trong `product([1, -1], repeat=n)`, tính `sum(s * x)`; nếu bằng `target` thì tăng biến đếm.

**Code demo:**

In [8]:
def ways_brute(nums, target):
    count = 0
    for signs in product([1, -1], repeat=len(nums)):
        if sum(s * x for s, x in zip(signs, nums)) == target:
            count += 1
    return count


for (nums, target), expected in examples:
    got = ways_brute(nums, target)
    print(nums, target, "->", got)
    assert got == expected
print(ways_brute([1, 2, 3], 0))      # 2, khớp bảng tính tay

[1, 1, 1, 1, 1] 3 -> 5
[1] 1 -> 1
2


**Code nộp LeetCode (Cách 1):**

In [9]:
class SolutionBrute:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        count = 0
        for signs in product((1, -1), repeat=len(nums)):
            if sum(s * x for s, x in zip(signs, nums)) == target:
                count += 1
        return count


for (nums, target), expected in examples:
    assert SolutionBrute().findTargetSumWays(nums, target) == expected
print("Cách 1 OK")

Cách 1 OK


**Độ phức tạp.** `2^n` bộ dấu, mỗi bộ tính tổng `O(n)` ⇒ **`O(n · 2^n)`** thời gian, `O(n)` bộ nhớ phụ (một bộ dấu). Với `n = 20`: khoảng 20 triệu phép — Python mất vài giây; vẫn có thể pass một số kiểm thử nhưng rủi ro.

**Nút thắt.** Ta **tính lại tổng từ đầu** cho từng bộ dấu, trong khi hai bộ dấu chung tiền tố có chung tổng một phần. Đệ quy sẽ dùng chung phần đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Chia nhỏ theo từng số.** Gọi `f(i, cur)` là số cách chọn dấu cho các số `nums[i:]` sao cho tổng cuối bằng `target`, biết tổng hiện tại là `cur`. Khi đó `f(i, cur) = f(i+1, cur + nums[i]) + f(i+1, cur - nums[i])`.

**Bước 2 — Tổng bị trùng lặp.** Số trạng thái khác nhau không phải `2^n` mà chỉ `n × (khoảng giá trị của cur)`. Vì `sum(nums) <= 1000`, `cur` thuộc `[-1000, 1000]` ⇒ tối đa `20 × 2001 ≈ 40 000` trạng thái. **Nhớ lại** các trạng thái này ⇒ Cách 3.

**Bước 3 — Đổi bài toán.** Gọi `P` (tổng nhóm `+`) và `N` (tổng nhóm `-`):

```
P - N = target
P + N = S          (S = tổng mọi số)
=> 2P = S + target => P = (S + target) / 2
```

Điều kiện có đáp án: `S + target` phải **chẵn** (nếu lẻ ⇒ 0 cách), và `|target| <= S` (nếu không thì `P` âm hoặc `> S` ⇒ 0 cách). Bài toán trở thành **đếm số tập con có tổng `P`** — knapsack 0-1 dạng đếm. Đây là Cách 4.

**Bước 4 — Hoặc giữ thẳng "tổng → số cách".** Duyệt từng số, giữ từ điển `ways[s]` = số cách gán dấu cho các số đã duyệt để được tổng `s`. Số mới `x`: `new[s + x] += ways[s]` và `new[s - x] += ways[s]`. Đây là Cách 5.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy `dfs(i, cur)` (chưa nhớ)

**Giải thích từng đoạn.**

1. `dfs(i, cur)`: đã xử lý `i` số đầu, tổng hiện tại `cur`.
2. Nếu `i == n`: trả `1` nếu `cur == target`, ngược lại `0`.
3. Ngược lại trả `dfs(i+1, cur + nums[i]) + dfs(i+1, cur - nums[i])`.

Cách này giống vét cạn nhưng **tính tổng dần** (không tính lại từ đầu mỗi lần) nên mỗi nút chỉ tốn `O(1)`.

In [10]:
def ways_dfs(nums, target):
    n = len(nums)

    def dfs(i, cur):
        if i == n:
            return 1 if cur == target else 0
        return dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])

    return dfs(0, 0)


for (nums, target), expected in examples:
    assert ways_dfs(nums, target) == expected
print(ways_dfs([1, 2, 3], 0))

2


In [11]:
class SolutionDFS:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        n = len(nums)

        def dfs(i: int, cur: int) -> int:
            if i == n:
                return 1 if cur == target else 0
            return dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])

        return dfs(0, 0)


for (nums, target), expected in examples:
    assert SolutionDFS().findTargetSumWays(nums, target) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp.** Cây nhị phân độ sâu `n` ⇒ `2^(n+1) - 1` lời gọi ⇒ **`O(2^n)`** thời gian, `O(n)` bộ nhớ ngăn xếp đệ quy. Nhanh hơn Cách 1 một hệ số `n` nhưng vẫn mũ.

### Cách 3 — Đệ quy có nhớ (top-down DP)

**Giải thích từng đoạn.** Giữ nguyên `dfs(i, cur)`, nhưng thêm `@lru_cache` để mỗi cặp `(i, cur)` chỉ tính **một lần**. Số trạng thái `i ∈ [0, n]`, `cur ∈ [-S, S]` ⇒ `O(n · S)`.

**Lưu ý nhỏ:** `lru_cache` lưu cache trong hàm; vì ta định nghĩa `dfs` bên trong method nên mỗi lần gọi method có cache riêng, không bị rò rỉ giữa các test.

In [12]:
def ways_memo(nums, target):
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i, cur):
        if i == n:
            return 1 if cur == target else 0
        return dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])

    return dfs(0, 0)


for (nums, target), expected in examples:
    assert ways_memo(nums, target) == expected
print(ways_memo([1, 2, 3], 0))
t0 = time.perf_counter()
print(ways_memo([1] * 20, 0), f"({time.perf_counter() - t0:.3f}s)")     # C(20, 10) = 184756

2
184756 (0.000s)


In [13]:
class SolutionMemo:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        n = len(nums)

        @lru_cache(maxsize=None)
        def dfs(i: int, cur: int) -> int:
            if i == n:
                return 1 if cur == target else 0
            return dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])

        return dfs(0, 0)


for (nums, target), expected in examples:
    assert SolutionMemo().findTargetSumWays(nums, target) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp.** `O(n · S)` trạng thái (`S = sum(nums)`), mỗi trạng thái `O(1)` ⇒ **`O(n · S)`** thời gian và **`O(n · S)`** bộ nhớ (cache + ngăn xếp `O(n)`). Với giới hạn đề: tối đa khoảng `20 × 2001` trạng thái.

### Cách 4 — Đổi sang "đếm tập con có tổng `P`" (knapsack 0-1 một chiều, chốt nộp)

**Giải thích từng đoạn.**

1. `S = sum(nums)`. Nếu `S + target < 0` hoặc `(S + target) % 2 == 1` (hoặc `target > S`) ⇒ trả `0`.
2. `P = (S + target) // 2`.
3. `dp[j]` = số tập con (của các số đã xét) có tổng đúng `j`. Khởi tạo `dp = [0] * (P + 1)`, `dp[0] = 1` (tập rỗng).
4. Với mỗi số `x`: cập nhật `dp[j] += dp[j - x]` cho `j` từ `P` **xuống** `x`. **Phải đi ngược** để `dp[j - x]` vẫn là giá trị **trước** khi xét `x` (nếu đi xuôi, `dp[j - x]` đã bao gồm `x` ⇒ `x` bị dùng nhiều lần).
5. Đáp án `dp[P]`.

**Trường hợp `x = 0`:** vòng `j` chạy đến `x = 0`: `dp[j] += dp[j - 0]` ⇒ `dp[j] *= 2` cho mọi `j`, đúng như "số `0` nhân đôi số cách". Vòng lặp `range(P, x - 1, -1)` với `x = 0` đi tới `j = 0` — nhớ bao gồm `j = x`.

In [14]:
def ways_knapsack(nums, target):
    S = sum(nums)
    if abs(target) > S or (S + target) % 2:
        return 0
    P = (S + target) // 2
    dp = [0] * (P + 1)
    dp[0] = 1
    for x in nums:
        for j in range(P, x - 1, -1):
            dp[j] += dp[j - x]
    return dp[P]


for (nums, target), expected in examples:
    assert ways_knapsack(nums, target) == expected
print(ways_knapsack([1, 2, 3], 0), ways_knapsack([0, 0, 1], 1), ways_knapsack([1, 2], 0))

2 4 0


In [15]:
class SolutionKnapsack:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        s = sum(nums)
        if abs(target) > s or (s + target) % 2:
            return 0
        p = (s + target) // 2
        dp = [0] * (p + 1)
        dp[0] = 1
        for x in nums:
            for j in range(p, x - 1, -1):
                dp[j] += dp[j - x]
        return dp[p]


for (nums, target), expected in examples:
    assert SolutionKnapsack().findTargetSumWays(nums, target) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp.** `O(n · P)` với `P <= S` ⇒ **`O(n · S)`** thời gian (tối đa `20 × 1000`), **`O(P)`** bộ nhớ — tốt nhất trong các cách DP vì chỉ một mảng một chiều.

### Cách 5 — DP bằng từ điển "tổng → số cách"

**Ý tưởng.** Không đổi bài toán; giữ trực tiếp các tổng có thể đạt. `ways = {0: 1}`. Với mỗi số `x`, tạo `new` mới: mỗi `(s, c)` trong `ways` đóng góp `c` cho `new[s + x]` và `new[s - x]`. Cuối cùng đáp án là `ways.get(target, 0)`.

Khác Cách 3 ở chỗ đây là DP **bottom-up theo từng số**, chỉ lưu các tổng **thực sự đạt được** (số khóa thường nhỏ hơn `2S + 1`).

In [16]:
def ways_dict(nums, target):
    ways = {0: 1}
    for x in nums:
        new = defaultdict(int)
        for s, c in ways.items():
            new[s + x] += c
            new[s - x] += c
        ways = new
    return ways.get(target, 0)


for (nums, target), expected in examples:
    assert ways_dict(nums, target) == expected
print(ways_dict([1, 2, 3], 0), ways_dict([0, 0, 1], 1))

2 4


In [17]:
class SolutionDict:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        ways = {0: 1}
        for x in nums:
            new = defaultdict(int)
            for s, c in ways.items():
                new[s + x] += c
                new[s - x] += c
            ways = new
        return ways.get(target, 0)


for (nums, target), expected in examples:
    assert SolutionDict().findTargetSumWays(nums, target) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp.** Sau mỗi số, số khóa tối đa `2S + 1` ⇒ **`O(n · S)`** thời gian; bộ nhớ **`O(S)`**. Với `x = 0`, `new[s+0]` và `new[s-0]` là cùng khóa nên được cộng hai lần `c` ⇒ nhân đôi, đúng.

## 8. TRACE từng bước

**Trace Cách 4** với `nums = [1, 1, 1, 1, 1]`, `target = 3`: `S = 5`, `P = (5 + 3) / 2 = 4`. Mảng `dp` có độ dài `P + 1 = 5` (chỉ số 0..4), ban đầu `[1, 0, 0, 0, 0]`.

| sau khi xét số thứ | `x` | `dp` (chỉ số 0..4) | ý nghĩa `dp[4]` |
|---|---|---|---|
| 0 (ban đầu) | — | `[1, 0, 0, 0, 0]` | 0 |
| 1 | 1 | `[1, 1, 0, 0, 0]` | 0 |
| 2 | 1 | `[1, 2, 1, 0, 0]` | 0 |
| 3 | 1 | `[1, 3, 3, 1, 0]` | 0 |
| 4 | 1 | `[1, 4, 6, 4, 1]` | 1 |
| 5 | 1 | `[1, 5, 10, 10, 5]` | **5** |

Đáp án `dp[4] = 5`. Các số của bảng chính là **hệ số nhị thức** `C(k, j)` vì có `k` số `1` giống nhau: chọn `j` trong `k` số để mang `+`.

Hàm `trace_knapsack` in bảng từng bước, `trace_dict` in từ điển của Cách 5.

In [18]:
def trace_knapsack(nums, target):
    S = sum(nums)
    print("nums =", nums, "target =", target, " S =", S)
    if abs(target) > S or (S + target) % 2:
        print("S + target lẻ hoặc |target| > S -> 0")
        return 0
    P = (S + target) // 2
    print("P = (S + target) // 2 =", P)
    dp = [0] * (P + 1)
    dp[0] = 1
    print("khởi tạo dp =", dp)
    for x in nums:
        for j in range(P, x - 1, -1):
            dp[j] += dp[j - x]
        print(f"sau x={x}: dp = {dp}")
    print("đáp án dp[P] =", dp[P])
    return dp[P]


trace_knapsack([1, 1, 1, 1, 1], 3)

nums = [1, 1, 1, 1, 1] target = 3  S = 5
P = (S + target) // 2 = 4
khởi tạo dp = [1, 0, 0, 0, 0]
sau x=1: dp = [1, 1, 0, 0, 0]
sau x=1: dp = [1, 2, 1, 0, 0]
sau x=1: dp = [1, 3, 3, 1, 0]
sau x=1: dp = [1, 4, 6, 4, 1]
sau x=1: dp = [1, 5, 10, 10, 5]
đáp án dp[P] = 5


5

In [19]:
def trace_dict(nums, target):
    ways = {0: 1}
    print("khởi tạo:", ways)
    for x in nums:
        new = defaultdict(int)
        for s, c in ways.items():
            new[s + x] += c
            new[s - x] += c
        ways = dict(sorted(new.items()))
        print(f"sau x={x}: {ways}")
    print(f"ways[{target}] =", ways.get(target, 0))
    return ways.get(target, 0)


trace_dict([1, 2, 3], 0)
print()
trace_dict([1, 0, 2], 3)          # có số 0: số cách nhân đôi

khởi tạo: {0: 1}
sau x=1: {-1: 1, 1: 1}
sau x=2: {-3: 1, -1: 1, 1: 1, 3: 1}
sau x=3: {-6: 1, -4: 1, -2: 1, 0: 2, 2: 1, 4: 1, 6: 1}
ways[0] = 2

khởi tạo: {0: 1}
sau x=1: {-1: 1, 1: 1}
sau x=0: {-1: 2, 1: 2}
sau x=2: {-3: 2, -1: 2, 1: 2, 3: 2}
ways[3] = 2


2

## 9. Code nộp LeetCode

Ta nộp Cách 4 (knapsack một chiều): ngắn, nhanh, ít bộ nhớ.

In [20]:
class Solution:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        s = sum(nums)
        # P - N = target, P + N = s  =>  P = (s + target) / 2
        if abs(target) > s or (s + target) % 2:
            return 0
        p = (s + target) // 2
        dp = [0] * (p + 1)
        dp[0] = 1
        for x in nums:
            for j in range(p, x - 1, -1):   # đi ngược: mỗi số dùng tối đa một lần
                dp[j] += dp[j - x]
        return dp[p]


for (nums, target), expected in examples:
    got = Solution().findTargetSumWays(nums, target)
    print(nums, target, "->", got)
    assert got == expected

[1, 1, 1, 1, 1] 3 -> 5
[1] 1 -> 1


## 10. Vì sao đúng

**Song ánh dấu ↔ tập con.** Mỗi cách gán dấu cho `n` số tương ứng **một-một** với tập con `A` ⊆ `{0..n-1}` là các chỉ số mang `+` (phần còn lại mang `-`). Hai cách gán dấu khác nhau cho hai tập con khác nhau (kể cả khi giá trị số trùng nhau, vì ta phân biệt theo **chỉ số**).

**Đại số.** Với tập con `A`: `P = Σ_{i∈A} nums[i]`, `N = S - P`. Giá trị biểu thức là `P - N = 2P - S`. Biểu thức bằng `target` ⇔ `2P - S = target` ⇔ `P = (S + target) / 2`. Nên đáp án = **số tập con có tổng `(S + target)/2`** (bằng 0 nếu giá trị đó không nguyên hoặc ngoài `[0, S]`).

**Knapsack 0-1.** Bất biến vòng ngoài: sau khi xét `k` số đầu, `dp[j]` = số tập con của `k` số đó có tổng `j`. Khi thêm số `x`: một tập con hoặc **không chứa** `x` (đã được tính trong `dp[j]` cũ) hoặc **chứa** `x` (đúng `dp_cũ[j - x]` tập con của các số trước cộng `x`) ⇒ `dp_mới[j] = dp_cũ[j] + dp_cũ[j - x]`. Duyệt `j` giảm dần đảm bảo `dp[j - x]` chưa bị ghi đè (vẫn là `dp_cũ`) khi ta đọc nó, vì `j - x < j` sẽ được cập nhật **sau** `j`.

**Số 0.** `x = 0` cho `dp_mới[j] = dp_cũ[j] + dp_cũ[j]` ⇒ nhân đôi, đúng vì `+0` và `-0` là hai biểu thức khác nhau.

## 11. Time / Space Complexity

Gọi `n = len(nums)`, `S = sum(nums) <= 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n · 2^n)` | `O(n)` |
| 2. Đệ quy thuần | `O(2^n)` | `O(n)` |
| 3. Đệ quy có nhớ | `O(n · S)` | `O(n · S)` |
| 4. Knapsack 1 chiều | `O(n · S)` | `O(S)` |
| 5. Từ điển tổng → cách | `O(n · S)` | `O(S)` |

Với `n = 20, S = 1000`: Cách 4 làm khoảng `20 000` phép cộng.

## 12. Bộ test

Oracle độc lập: `oracle(nums, target)` dùng `itertools.product` đếm trực tiếp (chính là Cách 1, nhưng được viết lại ở đây bằng **đệ quy sinh chuỗi biểu thức và `eval`** để không phụ thuộc vào đại số `P/N`). Với `n` nhỏ ta chạy cả `eval` thật; với `n` lớn hơn ta dùng tổng số học.

In [21]:
def oracle(nums, target):
    """Sinh từng biểu thức dạng chuỗi như '+2-1' rồi eval — hoàn toàn theo đề."""
    n = len(nums)
    count = 0
    for signs in product("+-", repeat=n):
        expr = "".join(s + str(x) for s, x in zip(signs, nums))
        if eval(expr) == target:
            count += 1
    return count


SOLUTIONS = [Solution(), SolutionBrute(), SolutionDFS(), SolutionMemo(), SolutionKnapsack(), SolutionDict()]
print(oracle([1, 1, 1, 1, 1], 3), oracle([1], 1), oracle([1, 2, 3], 0))

5 1 2


In [22]:
# 12.1 Ví dụ đề + edge case
cases = [
    ([1, 1, 1, 1, 1], 3, 5),
    ([1], 1, 1),
    ([1], -1, 1),
    ([1], 0, 0),                    # không thể = 0
    ([1], 2, 0),                    # |target| > S
    ([0], 0, 2),                    # +0 và -0
    ([0, 0, 0], 0, 8),              # 2^3
    ([0, 0, 1], 1, 4),              # hai số 0 nhân đôi 2 lần
    ([1, 2], 0, 0),                 # S + target lẻ
    ([1, 2, 3], 0, 2),
    ([1, 2, 3], 6, 1),
    ([1, 2, 3], -6, 1),
    ([1000], 1000, 1),              # biên giá trị
    ([1000], -1000, 1),
    ([1, 1], 0, 2),
    ([2, 2, 2, 2], 0, 6),           # C(4, 2)
]
for nums, target, expected in cases:
    assert oracle(nums, target) == expected, (nums, target, oracle(nums, target))
    for sol in SOLUTIONS:
        got = sol.findTargetSumWays(list(nums), target)
        assert got == expected, (type(sol).__name__, nums, target, got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 16 test x 6 lớp


In [23]:
# 12.2 Vét cạn mọi mảng độ dài <= 5 trên bảng {0,1,2,3}, mọi target trong [-S, S+1]
count = 0
for n in range(1, 6):
    for arr in product((0, 1, 2, 3), repeat=n):
        nums = list(arr)
        S = sum(nums)
        for target in range(-S - 1, S + 2):
            exp = oracle(nums, target) if n <= 4 else SolutionBrute().findTargetSumWays(nums, target)
            for sol in SOLUTIONS:
                assert sol.findTargetSumWays(nums, target) == exp, (type(sol).__name__, nums, target)
            count += 1
print("OK: đối chiếu", count, "cặp (nums, target)")

OK: đối chiếu 23208 cặp (nums, target)


In [24]:
# 12.3 Ngẫu nhiên có seed, đúng ràng buộc (sum <= 1000, target trong [-1000, 1000])
random.seed(494)
for _ in range(400):
    n = random.randint(1, 12)
    nums = [random.randint(0, 80) for _ in range(n)]
    if random.random() < 0.3:
        nums = [random.choice([0, 0, 1, 2]) for _ in range(n)]
    assert sum(nums) <= 1000
    S = sum(nums)
    if random.random() < 0.7:
        # chọn target "có khả năng": từ một cách gán dấu thật
        target = sum(random.choice((1, -1)) * x for x in nums)
    else:
        target = random.randint(-1000, 1000)
    exp = SolutionBrute().findTargetSumWays(nums, target)
    for sol in SOLUTIONS:
        assert sol.findTargetSumWays(nums, target) == exp, (type(sol).__name__, nums, target)
print("OK: 400 ca ngẫu nhiên n <= 12")

OK: 400 ca ngẫu nhiên n <= 12


In [25]:
# 12.4 Input lớn (n = 20): đối chiếu giữa các cách nhanh + giá trị đã biết
fast = [Solution(), SolutionMemo(), SolutionKnapsack(), SolutionDict()]
big = [
    ([1] * 20, 0, 184756),                 # C(20, 10)
    ([1] * 20, 20, 1),
    ([1] * 20, 18, 20),                    # C(20, 1)
    ([0] * 20, 0, 2 ** 20),                # mọi cách đều đúng
    ([50] * 20, 0, 184756),                # tổng = 1000
]
for nums, target, expected in big:
    t0 = time.perf_counter()
    got = [s.findTargetSumWays(nums, target) for s in fast]
    assert all(g == expected for g in got), (nums[:3], target, got, expected)
    print(f"n={len(nums)} target={target:>3} -> {expected:>7} ({time.perf_counter() - t0:.3f}s cho 4 cách)")

random.seed(4940)
for _ in range(100):
    n = 20
    nums = [random.randint(0, 100) for _ in range(n)]
    while sum(nums) > 1000:
        nums[random.randrange(n)] //= 2
    target = random.randint(-sum(nums), sum(nums))
    res = {s.findTargetSumWays(nums, target) for s in fast}
    assert len(res) == 1, (nums, target, res)
print("OK: 100 ca n = 20 ngẫu nhiên, các cách nhanh trùng nhau")

n=20 target=  0 ->  184756 (0.000s cho 4 cách)
n=20 target= 20 ->       1 (0.000s cho 4 cách)
n=20 target= 18 ->      20 (0.000s cho 4 cách)
n=20 target=  0 -> 1048576 (0.000s cho 4 cách)
n=20 target=  0 ->  184756 (0.001s cho 4 cách)


OK: 100 ca n = 20 ngẫu nhiên, các cách nhanh trùng nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: duyệt `j` xuôi ở knapsack 1 chiều (dùng một số nhiều lần)

In [26]:
def wrong_forward(nums, target):
    S = sum(nums)
    if abs(target) > S or (S + target) % 2:
        return 0
    P = (S + target) // 2
    dp = [0] * (P + 1)
    dp[0] = 1
    for x in nums:
        for j in range(x, P + 1):                 # SAI: đi xuôi
            dp[j] += dp[j - x]
    return dp[P]


nums, target = [1, 1, 1, 1, 1], 3
print("sai :", wrong_forward(nums, target))
print("đúng:", Solution().findTargetSumWays(nums, target))

sai : 70
đúng: 5


**Vì sao sai?** Khi đi xuôi, lúc tính `dp[j]` thì `dp[j - x]` **đã được cập nhật bằng chính số `x` này**, nên `x` bị dùng nhiều lần (bài toán "đếm cách dùng số không giới hạn" — complete knapsack). Mỗi số chỉ được mang đúng một dấu ⇒ phải đi **ngược** từ `P` xuống `x`.

### Lỗi 2: quên kiểm tra chẵn lẻ và `|target| > S`

In [27]:
def wrong_no_check(nums, target):
    S = sum(nums)
    P = (S + target) // 2                         # SAI: không kiểm tra S + target lẻ / âm
    if P < 0:
        return 0
    dp = [0] * (P + 1)
    dp[0] = 1
    for x in nums:
        for j in range(P, x - 1, -1):
            dp[j] += dp[j - x]
    return dp[P]


for nums, target in [([1, 2], 0), ([1, 1, 1], 2), ([1], 2)]:
    print(nums, target, "sai:", wrong_no_check(nums, target), " đúng:", Solution().findTargetSumWays(nums, target))

[1, 2] 0 sai: 1  đúng: 0
[1, 1, 1] 2 sai: 3  đúng: 0
[1] 2 sai: 1  đúng: 0


**Vì sao sai?** `[1, 2]` và `target = 0`: `S + target = 3` lẻ nên `P = 3/2` không nguyên ⇒ **không có cách**; `//` làm tròn xuống thành `1` và đếm sai tập con có tổng `1`. Ngoài ra khi `target > S`, `P > S` thì `dp[P]` vẫn có thể ngoài phạm vi ý nghĩa; hãy chặn sớm bằng `abs(target) > S` và `(S + target) % 2`.

### Lỗi 3: bỏ qua số `0` hoặc duyệt `j` dừng ở `x + 1`

In [28]:
def wrong_skip_zero(nums, target):
    S = sum(nums)
    if abs(target) > S or (S + target) % 2:
        return 0
    P = (S + target) // 2
    dp = [0] * (P + 1)
    dp[0] = 1
    for x in nums:
        if x == 0:                                # SAI: bỏ qua số 0
            continue
        for j in range(P, x - 1, -1):
            dp[j] += dp[j - x]
    return dp[P]


for nums, target in [([0, 0, 1], 1), ([0], 0), ([1, 0], 1)]:
    print(nums, target, "sai:", wrong_skip_zero(nums, target), " đúng:", Solution().findTargetSumWays(nums, target))

[0, 0, 1] 1 sai: 1  đúng: 4
[0] 0 sai: 1  đúng: 2
[1, 0] 1 sai: 1  đúng: 2


**Vì sao sai?** `+0` và `-0` là **hai biểu thức khác nhau** nên mỗi số `0` nhân đôi số cách. Bỏ qua `0` làm mất hệ số `2^{số lượng số 0}`. Vòng `for j in range(P, x - 1, -1)` với `x = 0` cộng `dp[j] += dp[j]` — chính là phép nhân đôi mong muốn; đừng viết `x` thành `x + 1` trong biên của `range`.

### Lỗi 4: memo thiếu khóa `cur`

In [29]:
def wrong_memo(nums, target):
    n = len(nums)
    memo = {}

    def dfs(i, cur):
        if i == n:
            return 1 if cur == target else 0
        if i in memo:                              # SAI: khóa chỉ là i
            return memo[i]
        memo[i] = dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])
        return memo[i]

    return dfs(0, 0)


nums, target = [1, 1, 1, 1, 1], 3
print("sai :", wrong_memo(nums, target))
print("đúng:", Solution().findTargetSumWays(nums, target))

sai : 16
đúng: 5


**Vì sao sai?** Kết quả `dfs(i, cur)` phụ thuộc cả `i` **lẫn** `cur` (tổng hiện tại). Khóa chỉ là `i` làm hai tổng khác nhau dùng chung đáp án. Khóa phải là cặp `(i, cur)`.

### Lỗi 5: dùng `set` các tổng đạt được thay vì đếm

Nếu chỉ lưu "tổng nào đạt được" thì ta trả lời được câu "có tồn tại cách nào không?" chứ không đếm được **bao nhiêu** cách — nhiều cách khác nhau cho cùng một tổng bị gộp thành một. Hãy dùng `dict` tổng → số cách (Cách 5).

## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "gán dấu `+`/`-` cho từng số", "chia mảng thành hai nhóm", `sum` nhỏ (`<= 1000`) ⇒ **subset sum / knapsack 0-1**.
- **Chuyển `+/-` thành tập con:** `P - N = target`, `P + N = S` ⇒ `P = (S + target) / 2`. Luôn nhớ hai điều kiện chặn: `abs(target) <= S` và `S + target` chẵn.
- **Knapsack 0-1 một chiều: duyệt dung lượng giảm dần.** Complete knapsack (dùng nhiều lần) thì duyệt tăng dần. Nhớ bằng câu: "đi ngược để không dùng lại".
- **Đếm khác với tồn tại:** `dp[j] = dp[j] or dp[j-x]` (boolean) cho "có thể?"; `dp[j] += dp[j-x]` cho "bao nhiêu cách?".
- **Số 0** luôn là bẫy đếm: mỗi số 0 nhân đôi.
- Khi `S` nhỏ mà `n` vừa phải, DP theo **giá trị tổng** thắng vét cạn theo `2^n`.

## 15. Biến thể

1. **Subset Sum / Partition Equal Subset Sum (416):** có tập con tổng `S/2` không? — knapsack boolean.
2. **Last Stone Weight II (1049):** chia các số thành hai nhóm với hiệu nhỏ nhất ⇒ cùng khung `P - N`.
3. **Ones and Zeroes (474):** knapsack 0-1 hai chiều.
4. **Expression Add Operators (282):** thêm cả toán tử `*` và ghép chữ số ⇒ không DP được, dùng backtracking.
5. **Liệt kê mọi biểu thức** thay vì đếm (cần backtracking; xem bài tập 5).
6. **Mỗi số có thể dùng nhiều lần (Coin Change 2, 518):** đổi chiều duyệt `j` thành xuôi.

In [30]:
# Biến thể 5: liệt kê mọi biểu thức đúng (backtracking)
def list_expressions(nums, target):
    n = len(nums)
    res = []

    def dfs(i, cur, expr):
        if i == n:
            if cur == target:
                res.append(expr)
            return
        dfs(i + 1, cur + nums[i], expr + "+" + str(nums[i]))
        dfs(i + 1, cur - nums[i], expr + "-" + str(nums[i]))

    dfs(0, 0, "")
    return res


for e in list_expressions([1, 1, 1, 1, 1], 3):
    print(e, "=", eval(e))

+1+1+1+1-1 = 3
+1+1+1-1+1 = 3
+1+1-1+1+1 = 3
+1-1+1+1+1 = 3
-1+1+1+1+1 = 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `product` | thử mọi bộ dấu | `O(n·2^n)` | `O(n)` | oracle, `n <= 12` |
| 2. Đệ quy thuần | `dfs(i, cur)` cộng/trừ | `O(2^n)` | `O(n)` | hiểu bản chất, `n <= 20` hơi chậm |
| 3. Đệ quy + memo | nhớ `(i, cur)` | `O(n·S)` | `O(n·S)` | tự nhiên nhất khi mới học DP |
| 4. Knapsack 1 chiều | đếm tập con có tổng `P` | `O(n·S)` | `O(S)` | **nộp bài**, nhanh và gọn |
| 5. Từ điển tổng → cách | DP bottom-up trên các tổng đạt được | `O(n·S)` | `O(S)` | không muốn biến đổi đại số, tổng thưa |

In [31]:
# So sánh thời gian thực nghiệm
nums = [1] * 20
print("nums = [1]*20, target = 0")
for name, sol in [("vét cạn", SolutionBrute()), ("đệ quy", SolutionDFS()), ("memo", SolutionMemo()),
                  ("knapsack", SolutionKnapsack()), ("dict", SolutionDict())]:
    t0 = time.perf_counter()
    r = sol.findTargetSumWays(nums, 0)
    print(f"{name:<9}: {r:>7} ({time.perf_counter() - t0:.4f}s)")

nums = [1]*20, target = 0


vét cạn  :  184756 (1.3908s)
đệ quy   :  184756 (0.1364s)
memo     :  184756 (0.0001s)
knapsack :  184756 (0.0000s)
dict     :  184756 (0.0001s)


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().findTargetSumWays([1, 1], 0)`; (b) `Solution().findTargetSumWays([0, 0], 0)`; (c) `Solution().findTargetSumWays([5], 3)`; (d) giá trị `P` khi `nums = [1, 2, 3, 4]`, `target = 2`.

**Bài 2 (tính tay).** Với `nums = [2, 3, 5]`, `target = 0`: tính `S`, `P`, rồi liệt kê các tập con có tổng `P` và các biểu thức tương ứng.

**Bài 3 (điền TODO).** Hoàn thành `ways_todo(nums, target)` theo Cách 4.

In [32]:
def ways_todo(nums, target):
    # TODO 1: s = sum(nums); trả 0 nếu abs(target) > s hoặc (s + target) lẻ
    # TODO 2: p = (s + target) // 2; dp = [0] * (p + 1); dp[0] = 1
    # TODO 3: với mỗi x trong nums: for j in range(p, x - 1, -1): dp[j] += dp[j - x]
    # TODO 4: return dp[p]
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `ways_memo2(nums, target)` theo Cách 3 (đệ quy có nhớ) **không nhìn lại** mục 7. Kiểm tra với `[1,1,1,1,1]`, `3` và `[0,0,1]`, `1`.

**Bài 5 (biến thể / mini challenge).** Viết `can_reach(nums, target)` trả `True/False`: **có tồn tại** cách gán dấu cho giá trị `target` hay không, bằng DP boolean (một `set` hoặc bitset). Kiểm chứng: `can_reach(nums, target) == (Solution().findTargetSumWays(nums, target) > 0)` trên nhiều ca ngẫu nhiên.

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ).** Bài 1 (d): `S = 10`, `P = (10 + 2) / 2`. Bài 3: chép đúng ba bước, nhớ vòng `j` đi **ngược**. Bài 5: tập các tổng đạt được `reach = {0}`; mỗi `x` thì `reach = {s + x for s in reach} | {s - x for s in reach}`.

**Mức 2 (vừa).** Bài 2: `S = 10`, `P = 5`. Tập con có tổng `5`: `{5}` và `{2, 3}`. Mỗi tập là nhóm mang dấu `+`.

**Mức 3 (sát đáp án).** Bài 4: `@lru_cache` trên `dfs(i, cur)`; điều kiện dừng `i == len(nums)`. Bài 5: kết quả `True` khi `target in reach` ở cuối.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
assert Solution().findTargetSumWays([1, 1], 0) == 2
assert Solution().findTargetSumWays([0, 0], 0) == 4
assert Solution().findTargetSumWays([5], 3) == 0
assert (sum([1, 2, 3, 4]) + 2) // 2 == 6

# Bài 2: nums=[2,3,5], target=0 -> S=10, P=5 -> {5}, {2,3} -> 2 cách
assert Solution().findTargetSumWays([2, 3, 5], 0) == 2
assert sorted(list_expressions([2, 3, 5], 0)) == ["+2+3-5", "-2-3+5"]
print("bài 1-2 OK")

bài 1-2 OK


In [34]:
# Bài 3
def ways_todo_answer(nums, target):
    s = sum(nums)
    if abs(target) > s or (s + target) % 2:
        return 0
    p = (s + target) // 2
    dp = [0] * (p + 1)
    dp[0] = 1
    for x in nums:
        for j in range(p, x - 1, -1):
            dp[j] += dp[j - x]
    return dp[p]


for (nums, target), expected in examples:
    assert ways_todo_answer(nums, target) == expected
print("bài 3 OK")

bài 3 OK


In [35]:
# Bài 4
def ways_memo2(nums, target):
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i, cur):
        if i == n:
            return int(cur == target)
        return dfs(i + 1, cur + nums[i]) + dfs(i + 1, cur - nums[i])

    return dfs(0, 0)


assert ways_memo2([1, 1, 1, 1, 1], 3) == 5
assert ways_memo2([0, 0, 1], 1) == 4
print("bài 4 OK")

bài 4 OK


In [36]:
# Bài 5
def can_reach(nums, target):
    reach = {0}
    for x in nums:
        reach = {s + x for s in reach} | {s - x for s in reach}
    return target in reach


random.seed(4941)
for _ in range(500):
    nums = [random.randint(0, 9) for _ in range(random.randint(1, 8))]
    target = random.randint(-sum(nums) - 2, sum(nums) + 2)
    assert can_reach(nums, target) == (Solution().findTargetSumWays(nums, target) > 0), (nums, target)
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (đếm tập con có tổng `P`)

```python
class Solution:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        s = sum(nums)
        if abs(target) > s or (s + target) % 2:
            return 0
        p = (s + target) // 2
        dp = [0] * (p + 1)
        dp[0] = 1
        for x in nums:
            for j in range(p, x - 1, -1):
                dp[j] += dp[j - x]
        return dp[p]
```

**Checklist**

- [ ] Hiểu: mỗi số có **đúng một** dấu; hai cách khác nhau nếu khác dấu ở **một vị trí** (kể cả hai số `0`)?
- [ ] Đã viết `P - N = target`, `P + N = S` ⇒ `P = (S + target) / 2`?
- [ ] Đã chặn `abs(target) > S` và `(S + target)` lẻ?
- [ ] `dp[0] = 1`, vòng `j` đi **ngược** từ `P` xuống `x` (bao gồm `x`)?
- [ ] Số `0` nhân đôi đúng (không `continue` bỏ qua)?
- [ ] Đã thử `[1]`, `[0]`, `[0,0,1]`, `target` âm, `target > S`, `n = 20`?
- [ ] Đã đối chiếu với vét cạn trên input nhỏ?